In [3]:
!pip install -q pypdf sentence-transformers faiss-cpu transformers sentencepiece gradio


In [4]:
from pypdf import PdfReader

pdf_path = "/content/Hospital_FAQ_Dataset.pdf"

reader = PdfReader(pdf_path)

hospital_text = ""

for page in reader.pages:
    text = page.extract_text()

    if text:
        hospital_text += text + "\n"

print("Pages:", len(reader.pages))
print("Characters:", len(hospital_text))

print("\nPreview:\n")
print(hospital_text[:3000])

Pages: 4
Characters: 8167

Preview:

Hospital FAQ Assistant - Knowledge Base
A clean, generic hospital FAQ dataset created for a RAG-based Hospital FAQ Assistant prototype. The
information is intentionally generic and should be replaced with verified hospital-specific information before
use in a real clinical setting.
1. What are the hospital visiting hours?
Answer: General visiting hours are from 4:00 PM to 6:00 PM. Visiting hours may vary by department.
2. Is emergency service available 24 hours?
Answer: Yes. The Emergency Department is available 24 hours a day, 7 days a week.
3. How can I contact the emergency department?
Answer: For emergencies, contact the hospital emergency desk or call the emergency number provided
by the hospital.
4. Is ambulance service available?
Answer: Yes. Ambulance services are available for patients who need transportation to or from the
hospital.
5. How can I book a doctor appointment?
Answer: You can book an appointment through the hospital reception, 

In [5]:
import re

hospital_text = hospital_text.replace("\x00", " ")

hospital_text = re.sub(
    r'\n\s*\n+',
    '\n\n',
    hospital_text
)

print(hospital_text[:3000])

Hospital FAQ Assistant - Knowledge Base
A clean, generic hospital FAQ dataset created for a RAG-based Hospital FAQ Assistant prototype. The
information is intentionally generic and should be replaced with verified hospital-specific information before
use in a real clinical setting.
1. What are the hospital visiting hours?
Answer: General visiting hours are from 4:00 PM to 6:00 PM. Visiting hours may vary by department.
2. Is emergency service available 24 hours?
Answer: Yes. The Emergency Department is available 24 hours a day, 7 days a week.
3. How can I contact the emergency department?
Answer: For emergencies, contact the hospital emergency desk or call the emergency number provided
by the hospital.
4. Is ambulance service available?
Answer: Yes. Ambulance services are available for patients who need transportation to or from the
hospital.
5. How can I book a doctor appointment?
Answer: You can book an appointment through the hospital reception, by phone, or through the hospital's
o

In [6]:
chunks = re.split(
    r'\n(?=\d+\.\s)',
    hospital_text
)

chunks = [
    chunk.strip()
    for chunk in chunks
    if len(chunk.strip()) > 30
]

print("Total chunks:", len(chunks))

print("\nFirst chunk:\n")
print(chunks[0])

Total chunks: 50

First chunk:

Hospital FAQ Assistant - Knowledge Base
A clean, generic hospital FAQ dataset created for a RAG-based Hospital FAQ Assistant prototype. The
information is intentionally generic and should be replaced with verified hospital-specific information before
use in a real clinical setting.


In [7]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


In [8]:
embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding shape: (50, 384)


In [9]:
import faiss
import numpy as np

embeddings = np.array(
    embeddings,
    dtype="float32"
)

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("FAISS database created!")
print("Number of chunks:", index.ntotal)
print("Number of chunks:", len(chunks))

FAISS database created!
Number of chunks: 50
Number of chunks: 50


In [10]:
def retrieve_context(question, top_k=3):

    question_embedding = embedding_model.encode(
        [question]
    )

    question_embedding = np.array(
        question_embedding,
        dtype="float32"
    )

    top_k = min(top_k, len(chunks))

    distances, indices = index.search(
        question_embedding,
        top_k
    )

    results = []

    for i in indices[0]:

        if 0 <= i < len(chunks):
            results.append(chunks[i])

    return results

In [11]:
question = "Is the pharmacy open 24 hours?"

results = retrieve_context(question, top_k=3)

print("Question:", question)

for i, result in enumerate(results, 1):

    print("\n--- Result", i, "---")
    print(result)

Question: Is the pharmacy open 24 hours?

--- Result 1 ---
17. Is the pharmacy open 24 hours?
Answer: The hospital pharmacy may provide 24-hour service depending on the facility. Please confirm
current operating hours with the pharmacy or reception.

--- Result 2 ---
16. Is a pharmacy available?
Answer: Yes. The hospital has a pharmacy where patients can obtain prescribed medicines.

--- Result 3 ---
2. Is emergency service available 24 hours?
Answer: Yes. The Emergency Department is available 24 hours a day, 7 days a week.


In [15]:
def ask_hospital(question):

    # Create question embedding
    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    # Create normalized chunk embeddings
    chunk_embeddings = embedding_model.encode(
        chunks,
        normalize_embeddings=True
    )

    # Calculate similarity
    similarities = np.dot(
        chunk_embeddings,
        question_embedding[0]
    )

    # Find best match
    best_index = np.argmax(similarities)
    best_score = similarities[best_index]

    print("Similarity:", round(float(best_score), 3))

    # Reject unrelated questions
    if best_score < 0.45:
        return "I couldn't find this information in the hospital FAQ."

    answer = chunks[best_index]

    # Remove question number
    answer = re.sub(
        r'^\d+\.\s*',
        '',
        answer
    )

    # Return only answer
    if "Answer:" in answer:
        answer = answer.split(
            "Answer:",
            1
        )[1]

    return answer.strip()

In [16]:
question = "What is the hospital helicopter service?"

print("Patient:", question)
print("Assistant:", ask_hospital(question))

Patient: What is the hospital helicopter service?
Similarity: 0.439
Assistant: I couldn't find this information in the hospital FAQ.


In [17]:
question = "Is the pharmacy open 24 hours?"

print("Patient:", question)
print("Assistant:", ask_hospital(question))

Patient: Is the pharmacy open 24 hours?
Similarity: 0.891
Assistant: The hospital pharmacy may provide 24-hour service depending on the facility. Please confirm
current operating hours with the pharmacy or reception.


In [18]:
import gradio as gr

# ==========================================
# CHATBOT FUNCTION
# ==========================================

def chatbot_response(message, history):
    return ask_hospital(message)


# ==========================================
# CUSTOM CSS
# ==========================================

custom_css = """

body {
    background: #eef6fb !important;
}

.gradio-container {
    max-width: 1200px !important;
    margin: auto !important;
    padding: 20px !important;
}

/* Header */

.header {
    background: linear-gradient(135deg, #075985, #0284c7, #06b6d4);
    color: white;
    padding: 30px;
    border-radius: 22px;
    text-align: center;
    margin-bottom: 18px;
    box-shadow: 0 10px 30px rgba(2,132,199,0.25);
}

.header-icon {
    font-size: 45px;
}

.header h1 {
    font-size: 34px;
    margin: 8px 0;
}

.header p {
    font-size: 16px;
    margin: 0;
}

/* Welcome */

.welcome {
    background: white;
    padding: 18px 22px;
    border-radius: 16px;
    border-left: 5px solid #0284c7;
    margin-bottom: 15px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.06);
}

.welcome-title {
    font-size: 19px;
    font-weight: 600;
    color: #075985;
}

.welcome-text {
    color: #64748b;
    margin-top: 6px;
}

/* Status */

.status {
    text-align: center;
    color: #15803d;
    font-size: 13px;
    margin: 12px;
}

/* Section */

.section-title {
    font-size: 17px;
    font-weight: 600;
    color: #075985;
    margin: 18px 0 8px 3px;
}

/* Chat */

.chat-container {
    border-radius: 18px !important;
    border: 1px solid #dbeafe !important;
    box-shadow: 0 5px 20px rgba(0,0,0,0.07) !important;
}

/* Footer */

.footer {
    text-align: center;
    margin-top: 18px;
    padding: 15px;
    color: #64748b;
    font-size: 12px;
}

.footer strong {
    color: #075985;
}
/* Search box */
.gradio-container textarea {
    background: white !important;
    color: #1e293b !important;
    border: 2px solid #0284c7 !important;
    border-radius: 15px !important;
    font-size: 16px !important;
    padding: 14px !important;
    box-shadow: 0 4px 12px rgba(0, 0, 0, 0.10) !important;
}

.gradio-container textarea::placeholder {
    color: #64748b !important;
    opacity: 1 !important;
}

.gradio-container textarea:focus {
    border: 2px solid #0369a1 !important;
    box-shadow: 0 0 0 3px rgba(2, 132, 199, 0.15) !important;
}

"""


# ==========================================
# BUILD INTERFACE
# ==========================================

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="cyan"
    ),
    css=custom_css,
    title="Hospital FAQ Assistant"
) as demo:

    # HEADER
    gr.HTML("""
    <div class="header">
        <div class="header-icon">🏥</div>
        <h1>Hospital FAQ Assistant</h1>
        <p>
            AI-powered assistant for quick hospital information
        </p>
    </div>
    """)

    # WELCOME
    gr.HTML("""
    <div class="welcome">
        <div class="welcome-title">
            👋 Welcome!
        </div>

        <div class="welcome-text">
            I can help you find information about appointments,
            emergency services, pharmacy, registration,
            visiting hours and other hospital services.
        </div>
    </div>
    """)

    # STATUS


    # QUICK QUESTIONS
    gr.HTML("""
    <div class="section-title">
        ⚡ Quick Questions
    </div>
    """)

    with gr.Row():

        q1 = gr.Button("💊 Pharmacy")

        q2 = gr.Button("📋 Registration")

        q3 = gr.Button("👨‍⚕️ Appointments")

        q4 = gr.Button("🚑 Emergency")


    # CHATBOT
    chatbot = gr.Chatbot(
        height=450,
        show_label=False,
        elem_classes="chat-container"
    )


    # CHAT INTERFACE
    chat = gr.ChatInterface(
        fn=chatbot_response,
        chatbot=chatbot,
        textbox=gr.Textbox(
            placeholder="🔍 Type your hospital question here...",
            show_label=False,
            lines=1
        ),
        examples=[
            "Is the pharmacy open 24 hours?",
            "What documents are needed for registration?",
            "How can I book a doctor appointment?",
            "Is emergency service available?",
            "What are the hospital visiting hours?"
        ],
        cache_examples=False
    )


    # QUICK BUTTONS
    q1.click(
        lambda: "Is the pharmacy open 24 hours?",
        outputs=chat.textbox
    )

    q2.click(
        lambda: "What documents are needed for registration?",
        outputs=chat.textbox
    )

    q3.click(
        lambda: "How can I book a doctor appointment?",
        outputs=chat.textbox
    )

    q4.click(
        lambda: "Is emergency service available?",
        outputs=chat.textbox
    )


    # FOOTER
    gr.HTML("""
    <div class="footer">
        <strong>🏥 Hospital FAQ Assistant</strong>
        <br><br>
        Powered by Sentence Transformers + FAISS + RAG
        <br><br>
        ⚠️ For general FAQ information only.
        For emergencies, contact hospital staff directly.
    </div>
    """)


# ==========================================
# LAUNCH
# ==========================================

demo.launch(share=True)

/tmp/ipykernel_494/3814873930.py:142: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b66ce48ae8ef8e0b68.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
